# AVTR-1 offline proof of concept

This notebook runs the upstream Linux/CUDA implementation on a Colab GPU and produces a short MP4. It is deliberately offline: the interactive WebRTC demo is a poor fit for Colab's proxied notebook environment.

Before running it, accept the model conditions at [avaturn-live/avtr-1](https://huggingface.co/avaturn-live/avtr-1), create a Hugging Face read token, and select **Runtime → Change runtime type → GPU**. L4 or A100 is preferred.

In [ ]:
# Check that Colab assigned an NVIDIA GPU.
import shutil
import subprocess

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No NVIDIA GPU found. In Colab choose Runtime > Change runtime type > GPU, then reconnect.")
subprocess.run(["nvidia-smi"], check=True)

## Install AVTR-1

This uses the project's locked pixi environment rather than modifying Colab's preinstalled Python packages. Re-running the cell updates an existing checkout instead of cloning over it.

In [ ]:
from pathlib import Path
import os

REPO = Path("/content/avtr-1")
PIXI = Path.home() / ".pixi/bin/pixi"
STORAGE = Path("/content/avtr1_storage")

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/avaturn-live/avtr-1.git", str(REPO)], check=True)

if not PIXI.exists():
    subprocess.run(["bash", "-lc", "curl -fsSL https://pixi.sh/install.sh | bash"], check=True)

os.environ["AVTR1_LOCAL_STORAGE"] = str(STORAGE)
subprocess.run([str(PIXI), "install"], cwd=REPO, check=True)

In [ ]:
# Verify CUDA from inside the locked environment and flag older GPUs.
probe = (
    "import torch; "
    "assert torch.cuda.is_available(), 'PyTorch cannot access CUDA'; "
    "p=torch.cuda.get_device_properties(0); "
    "print(f'GPU: {p.name} | compute capability: {p.major}.{p.minor} | VRAM: {p.total_memory/2**30:.1f} GiB'); "
    "assert (p.major,p.minor) >= (7,5), 'TensorRT 10 requires compute capability 7.5 or newer'; "
    "print('WARNING: upstream recommends Ampere (8.0+) or newer.' if (p.major,p.minor) < (8,0) else 'GPU meets the upstream architecture recommendation.')"
)
subprocess.run([str(PIXI), "run", "python", "-c", probe], cwd=REPO, check=True)

## Authenticate and download weights

The preferred route is a private Colab secret named `HF_TOKEN`. If it is absent, the cell asks through a hidden prompt. The token is kept out of notebook output.

In [ ]:
from getpass import getpass

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if not hf_token:
    hf_token = getpass("Hugging Face read token: ")
if not hf_token:
    raise RuntimeError("A Hugging Face token is required for the gated AVTR-1 files.")

env = os.environ.copy()
env["HF_TOKEN"] = hf_token
env["HF_HUB_TOKEN"] = hf_token
env["AVTR1_LOCAL_STORAGE"] = str(STORAGE)
subprocess.run(
    [str(PIXI), "run", "python", "scripts/download_artifacts.py"],
    cwd=REPO, env=env, check=True,
)

## Build the required GPU-specific TensorRT engines

Only the two AVTR-1 motion engines are mandatory. The renderer and HuBERT can use their CUDA ONNX Runtime fallbacks, which is sufficient for this offline proof of concept and avoids building five additional engines. Set `FULL_TENSORRT_BUILD = True` only if you want to spend extra setup time optimizing repeated renders. Engines must be rebuilt if Colab assigns a materially different GPU or the TensorRT environment changes.

In [ ]:
FULL_TENSORRT_BUILD = False
build_task = "build-trt-engines" if FULL_TENSORRT_BUILD else "build-trt-engines-avtr1"
print(f"Running {build_task!r}. The mandatory AVTR-1 build can still take several minutes.")
subprocess.run(
    [str(PIXI), "run", build_task],
    cwd=REPO, env=env, check=True,
)

## Capture a portrait with the laptop camera

Run the cell, allow camera access, frame a centered near-frontal face with visible shoulders, and press **Capture photo**. The cell center-crops the frame to the runtime's 1280×720 canvas without stretching it. If camera access is unavailable, it falls back to uploading a JPG, PNG, or WebP image. Only capture or upload a person's likeness with their explicit consent.

In [ ]:
import base64
from google.colab import files, output
from IPython.display import Audio, Image as DisplayImage, Video, display
from PIL import Image, ImageOps
from io import BytesIO

INPUT_DIR = Path("/content/avtr1_inputs")
INPUT_DIR.mkdir(parents=True, exist_ok=True)
PORTRAIT = INPUT_DIR / "custom_avatar.png"

camera_js = r"""
(async () => {
  const stream = await navigator.mediaDevices.getUserMedia({
    video: {facingMode: 'user', width: {ideal: 1280}, height: {ideal: 720}},
    audio: false
  });
  const panel = document.createElement('div');
  const video = document.createElement('video');
  const capture = document.createElement('button');
  video.srcObject = stream;
  video.autoplay = true;
  video.playsInline = true;
  video.style.display = 'block';
  video.style.width = '640px';
  video.style.maxWidth = '100%';
  video.style.transform = 'scaleX(-1)';
  capture.textContent = 'Capture photo';
  capture.style.marginTop = '8px';
  capture.style.padding = '8px 14px';
  panel.append(video, capture);
  document.body.appendChild(panel);
  await video.play();

  const dataUrl = await new Promise(resolve => {
    capture.onclick = () => {
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      resolve(canvas.toDataURL('image/jpeg', 0.95));
    };
  });
  stream.getTracks().forEach(track => track.stop());
  panel.remove();
  return dataUrl;
})()
"""

try:
    image_data_url = output.eval_js(camera_js)
    image_bytes = base64.b64decode(image_data_url.split(",", 1)[1])
    name = "camera capture"
except Exception as exc:
    print(f"Camera capture was unavailable ({exc}). Uploading a portrait instead.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No portrait was captured or uploaded.")
    name, image_bytes = next(iter(uploaded.items()))

with Image.open(BytesIO(image_bytes)) as source_image:
    source_image = ImageOps.exif_transpose(source_image).convert("RGB")
    portrait = ImageOps.fit(
        source_image, (1280, 720), method=Image.Resampling.LANCZOS, centering=(0.5, 0.5)
    )
    portrait.save(PORTRAIT, format="PNG")
CUSTOM_AVATAR_ID = "custom_avatar"
reference_dirs = list(STORAGE.glob("*/avatars_artifacts/reference_frames"))
if len(reference_dirs) != 1:
    raise RuntimeError(f"Expected one downloaded reference-frames directory, found: {reference_dirs}")
registered_portrait = reference_dirs[0] / f"{CUSTOM_AVATAR_ID}.png"
shutil.copy2(PORTRAIT, registered_portrait)
print(f"Prepared {name!r} and registered it as {CUSTOM_AVATAR_ID!r}")
display(DisplayImage(filename=str(PORTRAIT), width=640))

## Record speech

Run the cell, allow microphone access, speak, and press **Stop recording**. The browser recording is converted to mono 16 kHz WAV for AVTR-1. A short 3–10 second recording is ideal for the first test.

In [ ]:
import base64
from google.colab import output

recording_js = r"""
(async () => {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const preferred = 'audio/webm;codecs=opus';
  const options = MediaRecorder.isTypeSupported(preferred) ? {mimeType: preferred} : {};
  const recorder = new MediaRecorder(stream, options);
  const chunks = [];
  recorder.ondataavailable = event => { if (event.data.size > 0) chunks.push(event.data); };

  const panel = document.createElement('div');
  const status = document.createElement('span');
  const stop = document.createElement('button');
  status.textContent = 'Recording… ';
  stop.textContent = 'Stop recording';
  stop.style.padding = '8px 14px';
  panel.append(status, stop);
  document.body.appendChild(panel);

  const stopped = new Promise(resolve => { recorder.onstop = resolve; });
  stop.onclick = () => recorder.stop();
  recorder.start();
  await stopped;
  stream.getTracks().forEach(track => track.stop());
  panel.remove();

  const blob = new Blob(chunks, {type: recorder.mimeType || 'audio/webm'});
  return await new Promise(resolve => {
    const reader = new FileReader();
    reader.onloadend = () => resolve(reader.result);
    reader.readAsDataURL(blob);
  });
})()
"""
audio_data_url = output.eval_js(recording_js)
if not audio_data_url:
    raise RuntimeError("The browser did not return a recording.")

RAW_AUDIO = INPUT_DIR / "recording.webm"
AUDIO = INPUT_DIR / "recording.wav"
RAW_AUDIO.write_bytes(base64.b64decode(audio_data_url.split(",", 1)[1]))
subprocess.run(
    [str(PIXI), "run", "ffmpeg", "-y", "-i", str(RAW_AUDIO), "-ac", "1", "-ar", "16000", str(AUDIO)],
    cwd=REPO, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True,
)
print(f"Saved microphone recording to {AUDIO}")
display(Audio(filename=str(AUDIO)))

## Render the custom avatar

This preprocesses the captured face, drives it with the entire microphone recording, and muxes that recording into the resulting video. The defaults below are a conservative **lively** preset: stronger speech guidance and a wider stochastic-motion range, while preserving temporal smoothness. No TensorRT rebuild is needed. If artifacts appear, return `CFG_SELF_AUDIO` to `2.0` and `NOISE_TRUNC_Z` to `1.2`. This public renderer animates the head/face only; it cannot add true shoulder motion.

In [ ]:
# Natural defaults are CFG_SELF_AUDIO=2.0 and NOISE_TRUNC_Z=1.2.
# Raise these gradually; large values can cause pose jumps or face distortion.
CFG_SELF_AUDIO = 2.7
CFG_OTHER_AUDIO = 2.0
CFG_KP = 3.0
NOISE_ALPHA = 2.0       # Keep at 2.0 for smooth temporal motion.
NOISE_TRUNC_Z = 1.5     # Try at most ~1.7 before expecting artifacts.

# The upstream offline CLI does not expose RenderOptions, so load it and
# inject the runtime options before calling main().
render_wrapper = f"""
import importlib.util
from pathlib import Path
spec = importlib.util.spec_from_file_location('avtr1_generate_offline', Path('scripts/generate_offline.py'))
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
default_render_options = module.RenderOptions
def tuned_render_options(**kwargs):
    kwargs.update(
        cfg_self_audio={CFG_SELF_AUDIO}, cfg_other_audio={CFG_OTHER_AUDIO},
        cfg_kp={CFG_KP}, noise_alpha={NOISE_ALPHA}, noise_trunc_z={NOISE_TRUNC_Z},
    )
    return default_render_options(**kwargs)
module.RenderOptions = tuned_render_options
module.main()
"""

OUTPUT = Path("/content/avtr1_custom_poc.mp4")
subprocess.run(
    [
        str(PIXI), "run", "python", "-c", render_wrapper,
        "--avatar", CUSTOM_AVATAR_ID,
        "--speech", str(AUDIO),
        "--bg", "plain_white",
        "--out", str(OUTPUT),
    ],
    cwd=REPO, env=env, check=True,
)
assert OUTPUT.exists(), f"Expected output was not created: {OUTPUT}"

In [ ]:
display(Video(str(OUTPUT), embed=True))

# Uncomment to download the result to your Mac.
# files.download(str(OUTPUT))

## Optional: fully local conversational turn

The remaining cells add a no-API, turn-by-turn assistant: **laptop microphone → faster-whisper → Qwen3 → Chatterbox Turbo voice clone → AVTR-1 video**. All inference runs inside this Colab runtime. The models are downloaded from Hugging Face on first use, but no hosted inference API is called.

This is interactive in turns, not real-time streaming: after you speak, allow time for transcription, reply generation, voice synthesis, and then AVTR rendering. Use a clean 8–15 second recording in the earlier **Record speech** cell as the voice reference. Only clone a voice with permission.

In [ ]:
# Install the local assistant separately from AVTR's Pixi environment.
AGENT_VENV = Path("/content/avtr_local_agent_venv")
AGENT_PYTHON = AGENT_VENV / "bin/python"
AGENT_SCRIPT = Path("/content/avtr_local_turn.py")

venv_ready = AGENT_PYTHON.exists() and subprocess.run(
    [str(AGENT_PYTHON), "-m", "pip", "--version"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
).returncode == 0
if not venv_ready:
    if AGENT_VENV.exists():
        shutil.rmtree(AGENT_VENV)
    subprocess.run(
        [str(PIXI), "run", "python", "-m", "venv", str(AGENT_VENV)],
        cwd=REPO, env=env, check=True,
    )

subprocess.run(
    [str(AGENT_PYTHON), "-m", "pip", "install", "--upgrade", "pip"],
    check=True,
)
subprocess.run(
    [
        str(AGENT_PYTHON), "-m", "pip", "install",
        "chatterbox-tts==0.1.7",
        "faster-whisper>=1.1,<2",
        "accelerate>=1,<2",
    ],
    check=True,
)

In [ ]:
# Write the turn runner into /content. It loads one model at a time so
# an A100 or L4 has plenty of room left for the later AVTR render.
AGENT_SCRIPT.write_text(r'''
import argparse
import gc
import json
from pathlib import Path

import torch
import torchaudio as ta


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


parser = argparse.ArgumentParser()
parser.add_argument("--question-audio", required=True)
parser.add_argument("--voice-reference", required=True)
parser.add_argument("--output-audio", required=True)
parser.add_argument("--result-json", required=True)
parser.add_argument("--history", required=True)
parser.add_argument("--reset-history", action="store_true")
parser.add_argument("--stt-model", default="small.en")
parser.add_argument("--llm-model", default="Qwen/Qwen3-1.7B")
args = parser.parse_args()

if not torch.cuda.is_available():
    raise RuntimeError("The local conversational turn expects a Colab GPU runtime.")

# 1. Speech to text.
from faster_whisper import WhisperModel
stt = WhisperModel(args.stt_model, device="cuda", compute_type="float16")
segments, _ = stt.transcribe(args.question_audio, beam_size=5, vad_filter=True)
question = " ".join(segment.text.strip() for segment in segments).strip()
del stt
clear_memory()
if not question:
    raise RuntimeError("No speech was detected. Record the question again.")

# 2. Local conversational response.
from transformers import AutoModelForCausalLM, AutoTokenizer
history_path = Path(args.history)
history = [] if args.reset_history or not history_path.exists() else json.loads(history_path.read_text())
history = history[-8:]  # Keep at most four previous turns.
messages = [
    {"role": "system", "content": (
        "You are a friendly interactive avatar. Answer in one or two short, natural "
        "spoken sentences. Use plain text without markdown, lists, stage directions, or emoji."
    )},
    *history,
    {"role": "user", "content": question},
]
tokenizer = AutoTokenizer.from_pretrained(args.llm_model)
model = AutoModelForCausalLM.from_pretrained(
    args.llm_model, torch_dtype="auto", device_map="auto"
)
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)
inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
with torch.inference_mode():
    generated = model.generate(
        **inputs, max_new_tokens=96, do_sample=True, temperature=0.7,
        top_p=0.8, top_k=20, repetition_penalty=1.08,
    )
reply_ids = generated[0][inputs.input_ids.shape[-1]:]
reply = tokenizer.decode(reply_ids, skip_special_tokens=True).strip()
del generated, inputs, model, tokenizer
clear_memory()
if not reply:
    raise RuntimeError("The language model returned an empty reply.")

history.extend([
    {"role": "user", "content": question},
    {"role": "assistant", "content": reply},
])
history_path.write_text(json.dumps(history[-8:], indent=2))

# 3. Zero-shot voice cloning and speech synthesis.
from chatterbox.tts_turbo import ChatterboxTurboTTS
tts = ChatterboxTurboTTS.from_pretrained(device="cuda")
with torch.inference_mode():
    wav = tts.generate(reply, audio_prompt_path=args.voice_reference)
ta.save(args.output_audio, wav.cpu(), tts.sr)
Path(args.result_json).write_text(json.dumps({"question": question, "reply": reply}, indent=2))
print(json.dumps({"question": question, "reply": reply}, indent=2))
''')
print(f"Local turn runner ready: {AGENT_SCRIPT}")

### Record a question

This creates a new microphone recording without overwriting the earlier voice-reference clip. Run this cell again for each conversational turn.

In [ ]:
QUESTION_WEBM = INPUT_DIR / "question.webm"
QUESTION_WAV = INPUT_DIR / "question.wav"

question_recorder_js = r'''
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
const div = document.createElement('div');
const button = document.createElement('button');
const status = document.createElement('span');
button.textContent = 'Start recording question';
status.style.marginLeft = '12px';
div.appendChild(button);
div.appendChild(status);
document.body.appendChild(div);

const stream = await navigator.mediaDevices.getUserMedia({audio: true});
const recorder = new MediaRecorder(stream);
const chunks = [];
recorder.ondataavailable = event => chunks.push(event.data);
button.onclick = () => {
  if (recorder.state === 'inactive') {
    chunks.length = 0; recorder.start();
    button.textContent = 'Stop recording'; status.textContent = ' listening…';
  } else { recorder.stop(); button.disabled = true; status.textContent = ' processing…'; }
};
await new Promise(resolve => recorder.onstop = resolve);
stream.getTracks().forEach(track => track.stop());
const blob = new Blob(chunks, {type: recorder.mimeType || 'audio/webm'});
const reader = new FileReader();
reader.readAsDataURL(blob);
await new Promise(resolve => reader.onloadend = resolve);
div.remove();
reader.result;
'''

question_data_url = eval_js(question_recorder_js)
QUESTION_WEBM.write_bytes(base64.b64decode(question_data_url.split(',', 1)[1]))
subprocess.run(
    ["ffmpeg", "-y", "-i", str(QUESTION_WEBM), "-ac", "1", "-ar", "16000", str(QUESTION_WAV)],
    check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
display(Audio(str(QUESTION_WAV)))

### Generate the answer in your cloned voice

The first run downloads roughly several gigabytes of model weights and is much slower than later turns. Set `RESET_CHAT = True` whenever you want to discard the short conversation history.

In [ ]:
VOICE_REFERENCE = AUDIO  # The clean recording made earlier in the notebook.
REPLY_AUDIO = INPUT_DIR / "avatar_reply.wav"
TURN_RESULT = INPUT_DIR / "turn_result.json"
CHAT_HISTORY = INPUT_DIR / "conversation.json"
RESET_CHAT = False

turn_command = [
    str(AGENT_PYTHON), str(AGENT_SCRIPT),
    "--question-audio", str(QUESTION_WAV),
    "--voice-reference", str(VOICE_REFERENCE),
    "--output-audio", str(REPLY_AUDIO),
    "--result-json", str(TURN_RESULT),
    "--history", str(CHAT_HISTORY),
]
if RESET_CHAT:
    turn_command.append("--reset-history")

subprocess.run(turn_command, env=env, check=True)
turn = json.loads(TURN_RESULT.read_text())
print(f"You: {turn['question']}")
print(f"Avatar: {turn['reply']}")
display(Audio(str(REPLY_AUDIO)))

### Render the spoken answer

This reuses the portrait, avatar registration, and lively motion settings above. Repeat from **Record a question** for another turn.

In [ ]:
INTERACTIVE_OUTPUT = Path("/content/avtr1_interactive_turn.mp4")
subprocess.run(
    [
        str(PIXI), "run", "python", "-c", render_wrapper,
        "--avatar", CUSTOM_AVATAR_ID,
        "--speech", str(REPLY_AUDIO),
        "--bg", "plain_white",
        "--out", str(INTERACTIVE_OUTPUT),
    ],
    cwd=REPO, env=env, check=True,
)
display(Video(str(INTERACTIVE_OUTPUT), embed=True))